<a href="https://colab.research.google.com/github/EddyferO/Smart-Glass-Medical-Assistant/blob/main/Model_testing_py.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ── CELL 1: Install Dependencies ──────────────────────────────────────────
!pip install -q rouge-score nltk bert-score requests beautifulsoup4

SETUP

In [ ]:
# ── CELL 2: Upload File + Scrape All 100 Drug URLs ────────────────────────
import pandas as pd
import requests
from bs4 import BeautifulSoup
import time
import re
import io
from google.colab import files

print("Click the button below to upload your Excel file...")
uploaded = files.upload()

df = pd.read_excel(io.BytesIO(list(uploaded.values())[0]))
urls = df['URL'].tolist()
print(f"\nLoaded {len(urls)} URLs successfully\n")

headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'}

def extract_drug_text(url):
    try:
        r = requests.get(url.strip(), headers=headers, timeout=15)
        soup = BeautifulSoup(r.text, 'html.parser')
        for tag in soup(['script', 'style', 'nav', 'footer', 'header']):
            tag.decompose()
        text = soup.get_text(separator=' ', strip=True)
        text = re.sub(r'\s+', ' ', text)
        words = text.split()
        passage = ' '.join(words[:600])
        return passage, True
    except Exception as e:
        return str(e), False

print("Scraping URLs...\n")
drug_texts = []
failed = []

for i, url in enumerate(urls):
    text, success = extract_drug_text(url)
    if success and len(text.split()) > 50:
        drug_texts.append({"index": i+1, "url": url, "text": text})
        print(f"  [{i+1}/100] OK — {url[:70]}")
    else:
        failed.append({"index": i+1, "url": url})
        print(f"  [{i+1}/100] FAILED — {url[:70]}")
    time.sleep(0.3)

print(f"\n✓ Done — {len(drug_texts)} succeeded, {len(failed)} failed.")

Click the button below to upload your Excel file...


Saving 100 Drug URLs 2026_04_07.xlsx to 100 Drug URLs 2026_04_07.xlsx

Loaded 100 URLs successfully

Scraping URLs...

  [1/100] OK — https://labeling.pfizer.com/ShowLabeling.aspx?id=16474 
  [2/100] OK — https://dailymed.nlm.nih.gov/dailymed/fda/fdaDrugXsl.cfm?setid=ca73b51
  [3/100] OK — https://uspl.lilly.com/verzenio/verzenio.html#pi
  [4/100] OK — https://www.janssenlabels.com/package-insert/product-monograph/prescri
  [5/100] OK — https://dailymed.nlm.nih.gov/dailymed/fda/fdaDrugXsl.cfm?setid=75ddfd1
  [6/100] OK — https://dailymed.nlm.nih.gov/dailymed/fda/fdaDrugXsl.cfm?setid=1b16478
  [7/100] OK — https://dailymed.nlm.nih.gov/dailymed/fda/fdaDrugXsl.cfm?setid=afb118a
  [8/100] OK — https://dailymed.nlm.nih.gov/dailymed/drugInfo.cfm?setid=cecec8f6-1033
  [9/100] OK — https://dailymed.nlm.nih.gov/dailymed/fda/fdaDrugXsl.cfm?setid=9a368b3
  [10/100] OK — https://dailymed.nlm.nih.gov/dailymed/fda/fdaDrugXsl.cfm?setid=dbaf4bf
  [11/100] FAILED — https://docs.boehringer-ingelheim.com

In [ ]:
# ── CELL 3: Select 3 Showcase Drugs + Build All Test Cases ────────────────
showcase_indices = [0, 1, 13]

showcase_cases = [drug_texts[i] for i in showcase_indices if i < len(drug_texts)]
all_cases = drug_texts

print(f"Showcase cases  : {len(showcase_cases)}")
print(f"Total test cases: {len(all_cases)}")
for c in showcase_cases:
    print(f"  [{c['index']}] {c['url'][:70]}")

Showcase cases  : 3
Total test cases: 98
  [1] https://labeling.pfizer.com/ShowLabeling.aspx?id=16474 
  [2] https://dailymed.nlm.nih.gov/dailymed/fda/fdaDrugXsl.cfm?setid=ca73b51
  [15] https://dailymed.nlm.nih.gov/dailymed/fda/fdaDrugXsl.cfm?setid=91a0f73


MODEL 1 — LONG-T5

In [ ]:
# ── CELL 4: Free Memory, Load Long-T5, Summarize All Drugs ────────────────
import gc
import torch
import time
import warnings
import textwrap
from transformers import AutoTokenizer, LongT5ForConditionalGeneration

try:
    del model_mg
    del pipe_mg
    gc.collect()
    torch.cuda.empty_cache()
    print(f"GPU cleared. Free: {torch.cuda.mem_get_info()[0]/1e9:.2f} GB")
except:
    print("No MedGemma in memory, continuing...")

print("\nLoading Long-T5...")
tokenizer_lt5 = AutoTokenizer.from_pretrained("google/long-t5-tglobal-base")
model_lt5 = LongT5ForConditionalGeneration.from_pretrained(
    "google/long-t5-tglobal-base",
    torch_dtype=torch.float16
).to("cuda")
print("Long-T5 loaded!\n")

def summarize_long_t5(text, max_words=150):
    prompt = (
        "Generate a detailed abstractive summary of the following medical text. "
        "Do not copy sentences directly. Rewrite the key information in your own words, "
        "covering indications, dosing, warnings, side effects, and drug interactions: "
        + text
    )
    inputs = tokenizer_lt5(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=16384
    ).to(model_lt5.device)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        output_ids = model_lt5.generate(
            inputs["input_ids"],
            min_new_tokens=80,
            max_new_tokens=250,
            num_beams=4,
            length_penalty=2.0,
            no_repeat_ngram_size=4,
            early_stopping=False
        )
    return tokenizer_lt5.decode(output_ids[0], skip_special_tokens=True)

print("Running Long-T5 on all drugs (this will take a while, do not interrupt)...")
lt5_results = []
lt5_times   = []

for i, case in enumerate(all_cases):
    start = time.time()
    output = summarize_long_t5(case["text"])
    elapsed = round(time.time() - start, 2)
    lt5_results.append(output)
    lt5_times.append(elapsed)
    if (i + 1) % 10 == 0 or (i + 1) == len(all_cases):
        print(f"  Progress: {i+1}/{len(all_cases)} done...")

print(f"\n✓ Long-T5 done. Avg time: {round(sum(lt5_times)/len(lt5_times),2)}s")

No MedGemma in memory, continuing...

Loading Long-T5...


config.json:   0%|          | 0.00/851 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/297 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Long-T5 loaded!

Running Long-T5 on all drugs (this will take a while, do not interrupt)...
  Progress: 10/98 done...
  Progress: 20/98 done...
  Progress: 30/98 done...
  Progress: 40/98 done...
  Progress: 50/98 done...
  Progress: 60/98 done...
  Progress: 70/98 done...
  Progress: 80/98 done...
  Progress: 90/98 done...
  Progress: 98/98 done...

✓ Long-T5 done. Avg time: 7.09s


MODEL 2 — MEDGEMMA

In [ ]:
# ── CELL 5: Free Long-T5, Load MedGemma, Summarize All Drugs ──────────────
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

try:
    del model_lt5
except NameError:
    pass
gc.collect()
torch.cuda.empty_cache()
print(f"GPU cleared. Free: {torch.cuda.mem_get_info()[0]/1e9:.2f} GB")

print("\nLoading MedGemma...")
tokenizer_mg = AutoTokenizer.from_pretrained("google/medgemma-4b-it")
model_mg = AutoModelForCausalLM.from_pretrained(
    "google/medgemma-4b-it",
    dtype=torch.bfloat16,
    device_map="auto"
)
model_mg.generation_config.max_length = None
pipe_mg = pipeline(
    "text-generation",
    model=model_mg,
    tokenizer=tokenizer_mg,
    max_new_tokens=150  # reduced from 300
)
print("MedGemma loaded!\n")

def chunk_text(text, chunk_size=150):  # reduced from 200
    words = text.split()
    return [" ".join(words[i:i+chunk_size]) for i in range(0, len(words), chunk_size)]

def medgemma_summarize_chunk(chunk):
    messages = [{"role": "user", "content": (
        "Summarize this medical text in 2-3 sentences capturing key clinical info:\n\n"
        + chunk
    )}]
    formatted = tokenizer_mg.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        output = pipe_mg(formatted, return_full_text=False)
    return output[0]["generated_text"].strip()

def medgemma_base_prompt(text, max_words=150):
    chunks = chunk_text(text, chunk_size=150)
    if len(chunks) == 1:
        return medgemma_summarize_chunk(chunks[0])
    summaries = [medgemma_summarize_chunk(c) for c in chunks]
    combined  = " ".join(summaries)
    messages  = [{"role": "user", "content": (
        f"Combine these into one medical summary paragraph of {max_words} words "
        f"or less. No repeated information:\n\n" + combined
    )}]
    formatted = tokenizer_mg.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        output = pipe_mg(formatted, return_full_text=False)
    return output[0]["generated_text"].strip()

print("Running MedGemma on all drugs (do not interrupt)...")
mg_results = []
mg_times   = []

for i, case in enumerate(all_cases):
    if i > 0 and i % 5 == 0:  # clear every 5 instead of 10
        gc.collect()
        torch.cuda.empty_cache()

    start = time.time()
    output = medgemma_base_prompt(case["text"])
    elapsed = round(time.time() - start, 2)
    mg_results.append(output)
    mg_times.append(elapsed)

    if (i + 1) % 10 == 0 or (i + 1) == len(all_cases):
        free_gb = torch.cuda.mem_get_info()[0]/1e9
        print(f"  Progress: {i+1}/{len(all_cases)} done — GPU free: {free_gb:.2f} GB")

print(f"\n✓ MedGemma done. Avg time: {round(sum(mg_times)/len(mg_times),2)}s")

GPU cleared. Free: 41.84 GB

Loading MedGemma...


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

...

  Progress: 80/98 done — GPU free: 32.30 GB


Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both

  Progress: 90/98 done — GPU free: 32.13 GB


Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both

  Progress: 98/98 done — GPU free: 32.23 GB

✓ MedGemma done. Avg time: 31.84s


In [ ]:
# ── CELL 6: BERTScore + Showcase Summaries + Final Comparison ─────────────
import logging
import os
from bert_score import score as bert_score

logging.getLogger("transformers").setLevel(logging.ERROR)
os.environ["TOKENIZERS_PARALLELISM"] = "false"

def wrap_output(text, width=80, indent="  "):
    return "\n".join(textwrap.wrap(
        text.strip(), width=width,
        initial_indent=indent, subsequent_indent=indent
    ))

def get_bertscore_batch(hypotheses, references):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        _, _, F1 = bert_score(
            hypotheses, references, lang="en", verbose=False
        )
    return [round(f.item(), 4) for f in F1]

references = [c["text"] for c in all_cases]
print("Scoring all outputs with BERTScore...")
lt5_scores = get_bertscore_batch(lt5_results, references)
mg_scores  = get_bertscore_batch(mg_results,  references)
print("Done.\n")

# ── 3 Showcase Summaries ──────────────────────────────────────────────────
print("="*70)
print("  SHOWCASE — 3 EXAMPLE SUMMARIES")
print("="*70)

for case in showcase_cases:
    idx = next(i for i, c in enumerate(all_cases) if c["url"] == case["url"])

    print(f"\n{'─'*70}")
    print(f"  DRUG {idx+1}")
    print(f"  Source: {case['url'][:70]}")
    print(f"{'─'*70}")

    print(f"\n  ORIGINAL TEXT (first 150 words):")
    print(wrap_output(" ".join(case["text"].split()[:150])))

    print(f"\n  LONG-T5 SUMMARY:")
    print(wrap_output(lt5_results[idx]))
    print(f"\n  ⏱ Time: {lt5_times[idx]}s  |  BERTScore F1: {lt5_scores[idx]}")

    print(f"\n  MEDGEMMA SUMMARY:")
    print(wrap_output(mg_results[idx]))
    print(f"\n  ⏱ Time: {mg_times[idx]}s  |  BERTScore F1: {mg_scores[idx]}")

# ── Overall Comparison ────────────────────────────────────────────────────
avg_lt5_score = round(sum(lt5_scores) / len(lt5_scores), 4)
avg_mg_score  = round(sum(mg_scores)  / len(mg_scores),  4)
avg_lt5_time  = round(sum(lt5_times)  / len(lt5_times),  2)
avg_mg_time   = round(sum(mg_times)   / len(mg_times),   2)
winner        = "Long-T5"  if avg_lt5_score > avg_mg_score else "MedGemma"
faster        = "Long-T5"  if avg_lt5_time  < avg_mg_time  else "MedGemma"

print(f"\n{'='*70}")
print("  OVERALL RESULTS — 97 DRUG PRESCRIBING INFORMATION DOCUMENTS")
print(f"{'='*70}")
print(f"  {'Metric':<30} {'Long-T5':>10} {'MedGemma':>10}")
print(f"  {'─'*50}")
print(f"  {'Avg BERTScore F1':<30} {avg_lt5_score:>10} {avg_mg_score:>10}")
print(f"  {'Avg Time per Summary (s)':<30} {avg_lt5_time:>10} {avg_mg_time:>10}")
print(f"  {'Total Drugs Evaluated':<30} {len(all_cases):>10} {len(all_cases):>10}")
print(f"{'─'*70}")
print(f"  More Accurate Model : {winner}")
print(f"  Faster Model        : {faster}")
print(f"{'='*70}")

Scoring all outputs with BERTScore...


config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

Done.

  SHOWCASE — 3 EXAMPLE SUMMARIES

──────────────────────────────────────────────────────────────────────
  DRUG 1
  Source: https://labeling.pfizer.com/ShowLabeling.aspx?id=16474 
──────────────────────────────────────────────────────────────────────

  ORIGINAL TEXT (first 150 words):
  %PDF-1.4 %���� 151 0 obj < > endobj xref 151 49 0000000016 00000 n 0000001371
  00000 n 0000001535 00000 n 0000002456 00000 n 0000002872 00000 n 0000002924
  00000 n 0000002976 00000 n 0000003149 00000 n 0000003326 00000 n 0000003494
  00000 n 0000003668 00000 n 0000003794 00000 n 0000003957 00000 n 0000004082
  00000 n 0000004109 00000 n 0000004588 00000 n 0000004742 00000 n 0000004812
  00000 n 0000005053 00000 n 0000005576 00000 n 0000005799 00000 n 0000006309
  00000 n 0000006336 00000 n 0000006632 00000 n 0000006787 00000 n 0000006857
  00000 n 0000007086 00000 n 0000007123 00000 n 0000007359 00000 n 0000007628
  00000 n 0000007856 00000 n 0000008201 00000 n 0000008439 00000 n 0000008462
  